In [ ]:
import os
import sys

In [1]:
!git clone https://github.com/shelljane/lithobench
!git clone https://github.com/cuhk-eda/neural-ilt

Cloning into 'lithobench'...
remote: Enumerating objects: 902, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 902 (delta 30), reused 23 (delta 23), pack-reused 867 (from 1)
Receiving objects: 100% (902/902), 14.09 MiB | 12.50 MiB/s, done.
Resolving deltas: 100% (149/149), done.
Cloning into 'neural-ilt'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 130 (delta 0), reused 1 (delta 0), pack-reused 127 (from 1)
Receiving objects: 100% (130/130), 101.01 MiB | 29.29 MiB/s, done.
Resolving deltas: 100% (27/27), done.


In [2]:
%pip install -r lithobench/requirements_pip.txt

Note: you may need to restart the kernel to use updated packages.


In [ ]:
%cd /home/murilo/Documentos/Lithography/lithobench

!python3 lithobench/train.py -m lithobench/ilt/neuralilt.py -a NeuralILT -i 512 -t ILT -o dev -s MetalSet -n 8 -b 12 -p True

/home/murilo/Documentos/Lithography/lithobench
Training set: 14825, Test set: 1647
[Pre-Epoch 0] Training
100%|██████████████████████████| 1236/1236 [05:48<00:00,  3.54it/s, loss=0.0454]
[Pre-Epoch 0] Testing
100%|████████████████████████████| 138/138 [00:13<00:00,  9.92it/s, loss=0.0934]
[Pre-Epoch 0] loss = 0.0949229362110297
[Pre-Epoch 1] Training
100%|██████████████████████████| 1236/1236 [05:49<00:00,  3.54it/s, loss=0.0414]
[Pre-Epoch 1] Testing
100%|████████████████████████████| 138/138 [00:13<00:00, 10.10it/s, loss=0.0459]
[Pre-Epoch 1] loss = 0.04664165858665238
[Pre-Epoch 2] Training
 10%|██▊                        | 129/1236 [00:37<05:12,  3.55it/s, loss=0.0398]

In [ ]:
%cd /home/murilo/Documentos/Lithography/lithobench

# Bugfix: checkpoint salvo em work/, nao saved/; garante CWD correto no kernel.
!python3 lithobench/test.py -m NeuralILT -s MetalSet -l work/MetalSet_NeuralILT/net.pth --shots

In [ ]:
# Geracao do heatmap de hotspots a partir do PVBand (NeuralILT)
# Hotspot = regiao com alta variacao entre litho nominal/inner/outer (PVBand alto).
import glob, os
import numpy as np
import torch
import matplotlib.pyplot as plt
from lithobench.ilt.neuralilt import NeuralILT
from lithobench import evaluate
import pylitho.simple as lithosim

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CKPT   = 'work/MetalSet_NeuralILT/net.pth'
OUTDIR = 'work/hotspots'
os.makedirs(OUTDIR, exist_ok=True)

model = NeuralILT(size=512)
model.load(CKPT)

targets = evaluate.getTargets(samples=4, dataset='MetalSet')
litho = lithosim.LithoSim('./config/lithosimple.txt').to(DEVICE)

for idx, tgt in enumerate(targets):
    tgt_t = torch.from_numpy(tgt).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
    mask  = model.run(tgt_t)
    nom, inn, out = litho(mask)
    pvband = (out - inn).abs().squeeze().detach().cpu().numpy()
    plt.imsave(f'{OUTDIR}/hotspot_{idx}.png', pvband, cmap='hot')
    print(f'[{idx}] PVBand sum = {pvband.sum():.1f}')
